# Preparing Seismic Data for Earthquake–Explosion Classification

## 1. Load the metadata

The [PNW metadata](https://dasway.ess.washington.edu/shared/niyiyu/PNW-ML/comcat_metadata.csv) describes station recordings. Several recordings can belong to the same event.

We clean this metadata, select recordings with saved waveform measurements, and prepare one row per event. The [cleaning functions](../data_processing/clean_data.py) contain the rules used below.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Run from notebooks/. Its parent contains data/ and data_processing/.
ROOT = Path("..").resolve()
# Make the project's cleaning functions available to this notebook.
sys.path.insert(0, str(ROOT))
from data_processing.clean_data import (
    FEATURES, clean_labels, clean_context, select_recordings,
    aggregate_features, make_event_table,
)

DATA = ROOT / "data"
raw = pd.read_csv(DATA / "comcat_metadata.csv", dtype={
    "event_id": "string", "station_network_code": "string", "station_code": "string",
})

In [2]:
print(f"{len(raw):,} recordings from {raw['event_id'].nunique():,} events")
raw.head()

178,939 recordings from 46,054 events


,event_id,source_origin_time,source_latitude_deg,source_longitude_deg,source_type,source_depth_km,preferred_source_magnitude,preferred_source_magnitude_type,preferred_source_magnitude_uncertainty,source_depth_uncertainty_km,source_horizontal_uncertainty_km,station_network_code,station_channel_code,station_code,station_location_code,station_latitude_deg,station_longitude_deg,station_elevation_m,trace_name,trace_sampling_rate_hz,trace_start_time,trace_S_arrival_sample,trace_P_arrival_sample,trace_S_arrival_uncertainty_s,trace_P_arrival_uncertainty_s,trace_P_polarity,trace_S_onset,trace_P_onset,trace_snr_db,source_type_pnsn_label,source_local_magnitude,source_local_magnitude_uncertainty,source_duration_magnitude,source_duration_magnitude_uncertainty,source_hand_magnitude,trace_missing_channel,trace_has_offset
0,uw10564613,2002-10-03T01:56:49.530000Z,48.553,-122.520,earthquake,14.907,2.1,md,0.03,1.68,0.694,UW,BH,GNW,--,47.564,-122.825,220.0,"bucket4$0,:3,:15001",100,2002-10-03T01:55:59.530000Z,8097,6733,0.04,0.02,undecidable,NaN,NaN,6.135|3.065|11.766,eq,NaN,NaN,2.1,0.03,NaN,0,1
1,uw10564613,2002-10-03T01:56:49.530000Z,48.553,-122.520,earthquake,14.907,2.1,md,0.03,1.68,0.694,UW,EH,RPW,--,48.448,-121.515,850.0,"bucket9$0,:3,:15001",100,2002-10-03T01:55:59.530000Z,7258,6238,0.04,0.01,undecidable,NaN,NaN,nan|nan|22.583,eq,NaN,NaN,2.1,0.03,NaN,2,0
2,uw10564613,2002-10-03T01:56:49.530000Z,48.553,-122.520,earthquake,14.907,2.1,md,0.03,1.68,0.694,UW,BH,SQM,--,48.074,-123.048,45.0,"bucket4$1,:3,:15001",100,2002-10-03T01:55:59.530000Z,7037,6113,0.08,0.04,undecidable,NaN,NaN,1.756|3.057|3.551,eq,NaN,NaN,2.1,0.03,NaN,0,1
3,uw10564613,2002-10-03T01:56:49.530000Z,48.553,-122.520,earthquake,14.907,2.1,md,0.03,1.68,0.694,UW,EH,MCW,--,48.679,-122.833,693.0,"bucket7$0,:3,:15001",100,2002-10-03T01:55:59.530000Z,5894,5442,0.04,0.01,negative,NaN,NaN,nan|nan|27.185,eq,NaN,NaN,2.1,0.03,NaN,2,0
4,uw10568748,2002-09-26T07:00:04.860000Z,48.481,-123.133,earthquake,22.748,2.9,md,0.03,0.91,0.672,UW,HH,SP2,--,47.556,-122.249,30.0,"bucket3$0,:3,:15001",100,2002-09-26T06:59:14.860000Z,8647,6948,0.04,0.01,negative,NaN,NaN,10.881|17.107|2.242,eq,NaN,NaN,2.9,0.03,NaN,0,1


## 2. Keep the earthquake and explosion labels

Earthquakes receive label 0 and explosions receive label 1. PNSN `eq` maps to 0; `px`, `ex`, and `qb` map to 1. We use the generic label only when the PNSN code is empty. For example, a blank PNSN code with `earthquake` becomes 0; an unsupported PNSN code with `earthquake` is excluded.

If one recording has an unsupported or conflicting label, every recording of that event is removed. A missing event, network, station, or trace identifier removes that individual row.

In [3]:
labeled = clean_labels(raw)
print(f"{len(labeled):,} recordings after label and identity cleaning")

177,735 recordings after label and identity cleaning


## 3. Clean the required context

Each event needs valid, consistent time, coordinates, magnitude, and depth. We compare these values across its recordings and exclude the whole event if a required value is invalid or inconsistent. Negative depths and missing optional uncertainty fields remain.

Publisher missing-component and offset flags remove individual recordings. An event can remain when another station has a usable recording.

In [4]:
candidates = clean_context(labeled)
print(f"{len(candidates):,} metadata candidates")

124,112 metadata candidates


## 4. Select station recordings

The saved waveform QC table identifies recordings by their original CSV row number. We join those results to the metadata and keep recordings marked `pass`.

When several streams pass at one station, we keep the first trace name in sorted order. Selection after QC allows a valid stream to survive when another stream at that station fails. The saved event list supplies the completed identity review and geographic folds.

In [5]:
waveform_results = pd.read_csv(
    DATA / "waveform_measurements.csv.gz", usecols=["source_row", "status"]
)
saved_groups = pd.read_csv(DATA / "event_groups_and_folds.csv.gz")
recordings = select_recordings(candidates, waveform_results, saved_groups)
print(f"{len(recordings):,} selected recordings from {recordings['event_id'].nunique():,} events")

124,111 selected recordings from 38,581 events


## 5. Prepare the event features

We group the selected station measurements by event ID and calculate a median and count for each feature. A missing measurement affects only that feature; zero remains valid. The [feature definitions](../docs/feature_definitions.csv) describe the five saved measurements and their units.

The earlier event checkpoint supplies coordinates, magnitude, depth, local-time features, crest factor, and spectral entropy. Waveform extraction was completed separately. Changes to raw context or station selection require updated checkpoint values and measurements.

When building the event table, we match labels, folds, and measurements by event ID. For example, measurements ordered B, A still attach to events A, B correctly. The checkpoint's event order is retained in the CSV.

In [6]:
measurements = pd.read_csv(
    DATA / "recording_features.csv.gz",
    usecols=["source_row", "event_id", "station_id"] + FEATURES,
    float_precision="round_trip",  # Keep the saved measurement precision.
)
medians, station_counts = aggregate_features(recordings, measurements)
medians.head()

,nb_log10_pz_sh_rms_ratio_4_8hz,nb_log10_sh_sz_rms_ratio_4_8hz,nb_envelope_rise_10_90_s_2_10hz,nb_envelope_excess_kurtosis_2_10hz,nb_log10_psd_power_1_5_over_5_15hz
event_id,,,,,
uw10547793,-1.122150,1.040345,4.07,10.797078,-0.969816
uw10547878,0.154676,0.480488,1.45,4.102076,0.431430
uw10549468,-0.155038,0.260246,4.63,1.901415,-0.143798
uw10549578,-1.428289,0.300007,14.50,3.993996,-0.263133
uw10549668,-0.538502,0.316037,12.15,2.513362,0.090698


In [7]:
# Keep the checkpoint fields as their saved text.
checkpoint = pd.read_csv(DATA / "event_checkpoint.csv.gz", dtype=str, keep_default_na=False)
cleaned = make_event_table(checkpoint, recordings, medians, station_counts, saved_groups)
print(f"{len(cleaned):,} events before the final exclusions")

38,581 events before the final exclusions


## 6. Save the cleaned CSV

Events `uw10757783` and `uw61857197` have no station support for either amplitude feature because their usable P-windows are shorter than 0.3 seconds. We remove these two events and save the remaining 38,579 rows.

`label` is the classification target. Identifiers, geographic folds, and station counts support joins, evaluation, and data-quality review.

In [8]:
exclusions = pd.read_csv(DATA / "excluded_events.csv.gz")
event_is_excluded = cleaned["event_id"].isin(exclusions["event_id"])
cleaned = cleaned.loc[~event_is_excluded]
exclusions[["event_id", "reason"]]

,event_id,reason
0,uw10757783,p_window_under_0p3s
1,uw61857197,p_window_under_0p3s


In [9]:
assert cleaned["event_id"].is_unique
assert cleaned.shape == (38579, 22)

output_folder = ROOT / "deliverables"
output_folder.mkdir(exist_ok=True)
cleaned.to_csv(output_folder / "pnw_training_corrected.csv", index=False, lineterminator="\n")
print(f"Saved {len(cleaned):,} events and {cleaned.shape[1]} columns")

Saved 38,579 events and 22 columns


## 7. How the geographic groups were made

This optional section uses the 38,581 identity-reviewed events before the two final exclusions. Events begin in approximate 20 km cells. A pair within 30 seconds and 25 km connects its cells. If A connects to B and B connects to C, all three cells share a group.

Five-fold stratified group splitting keeps each group together while trying to balance the labels. These groups are geographic proxies, with no additional spatial buffer. The [grouping explanation](../docs/groups_and_folds.md) walks through the code and its original sources.

We align the reconstructed and saved tables by event ID and compare their assignments. The CSV above continues to use the saved assignments.

In [10]:
from data_processing.make_groups import geographic_groups, geographic_folds

columns = [
    "event_id", "label", "source_origin_time",
    "source_latitude_deg", "source_longitude_deg",
]
event_was_reviewed = candidates["event_id"].isin(saved_groups["event_id"])
reviewed_events = candidates.loc[event_was_reviewed, columns]
reviewed_events = reviewed_events.drop_duplicates("event_id")
rebuilt, nearby_pairs = geographic_groups(reviewed_events)
rebuilt = geographic_folds(rebuilt)

print(f"{len(rebuilt):,} events, {rebuilt['source_cell'].nunique():,} cells")
print(f"{len(nearby_pairs):,} nearby pairs, {rebuilt['spatial_group'].nunique():,} geographic groups")

38,581 events, 1,108 cells
134 nearby pairs, 1,100 geographic groups


In [11]:
comparison_columns = ["source_cell", "spatial_group", "geographic_fold"]
# Use the same event order before comparing each value.
reconstructed = rebuilt.set_index("event_id")[comparison_columns].sort_index()
saved = saved_groups.set_index("event_id")[comparison_columns].sort_index()
assert reconstructed.index.tolist() == saved.index.tolist()
matching_values = reconstructed.eq(saved)
matching_values.all().rename("matches saved assignments")

source_cell        True
spatial_group      True
geographic_fold    True
Name: matches saved assignments, dtype: bool

In [12]:
events_by_fold_and_label = rebuilt.groupby(["geographic_fold", "label"])
counts_by_fold_and_label = events_by_fold_and_label.size()
# Turn label 0 and label 1 into separate columns, with folds as the rows.
fold_counts = counts_by_fold_and_label.unstack(fill_value=0)
fold_counts.columns = ["earthquakes", "explosions"]
fold_counts

,earthquakes,explosions
geographic_fold,,
1,6587,1138
2,6584,1135
3,6581,1132
4,6581,1132
5,6580,1131
